In [19]:
import plotly.graph_objects as go
from ipywidgets import Output, VBox, HTML, FloatRangeSlider, Button
from IPython.display import display
import csv
from deap_optim import get_results

# Виджет вывода параметров
output_area = Output()

# Выбранная точка для сохранения
selected_point = {}

# График
fig = go.FigureWidget()

In [20]:
populations, pareto_front = get_results()

In [22]:
# Все индивиды
fig.add_trace(go.Scatter(
    x=populations[0], y=populations[1],
    mode='markers',
    marker=dict(color='royalblue', size=6, opacity=0.7),
    name='Individuals',
    customdata=populations[2],
    hovertemplate="Макс. нагрузка: %{x:.2f}<br>Эффективность: %{y:.2f}"
))

# Парето фронт
fig.add_trace(go.Scatter(
    x=pareto_front[0], y=pareto_front[1],
    mode='lines+markers',
    marker=dict(color='crimson', size=10),
    line=dict(dash='dash'),
    name='Pareto front',
    customdata=pareto_front[2],
    hovertemplate="Макс. нагрузка: %{x:.2f}<br>Эффективность: %{y:.2f}"
))

fig.update_layout(
    title="🎯 Парето-фронт и популяция решений",
    xaxis_title='Макс. нагрузка (%)',
    yaxis_title='Эффективность (%)',
    margin=dict(l=30, r=30, t=40, b=20),
    hovermode='closest'
)

# Обработка клика по точке
def handle_click(trace, points, selector):
    if points.point_inds:
        idx = points.point_inds[0]
        params = trace.customdata[idx]
        selected_point['data'] = params

        # HTML-таблица
        rows = ''.join([f"<tr><td>Параметр {i+1}</td><td>{val}</td></tr>" for i, val in enumerate(params)])
        html_table = f"""
        <table style="border-collapse: collapse; margin-top: 5px;">
          <thead><tr><th style="padding:4px;border:1px solid #ccc;">Параметр</th>
                     <th style="padding:4px;border:1px solid #ccc;">Значение</th></tr></thead>
          <tbody>{rows}</tbody>
        </table>
        """

        with output_area:
            output_area.clear_output()
            display(HTML(f"<b>Выбрана точка ({trace.name})</b>" + html_table))
        save_button.disabled = False

fig.data[0].on_click(handle_click)
fig.data[1].on_click(handle_click)


def save_solution(_):
    if 'data' in selected_point:
        with open('selected_solution.csv', 'w', newline='') as f:
            writer = csv.writer(f)
            writer.writerow([f'param_{i+1}' for i in range(len(selected_point['data']))])
            writer.writerow(selected_point['data'])
        with output_area:
            print("Решение сохранено в файл selected_solution.csv")

save_button = Button(description="Сохранить решение", disabled=True, button_style='success')

save_button.on_click(save_solution)

eff_slider = FloatRangeSlider(
    value=[20, 90],
    min=0, max=100, step=1,
    description='Эффективность:',
    continuous_update=False,
    layout=dict(width='60%')
)

def apply_filter(change=None):
    min_eff, max_eff = eff_slider.value
    y_vals = populations[1]
    visible = [(min_eff <= y <= max_eff) for y in y_vals]
    new_x = [x for x, show in zip(populations[0], visible) if show]
    new_y = [y for y, show in zip(populations[1], visible) if show]
    new_data = [d for d, show in zip(populations[2], visible) if show]

    with fig.batch_update():
        fig.data[0].x = new_x
        fig.data[0].y = new_y
        fig.data[0].customdata = new_data

eff_slider.observe(apply_filter, names='value')

instruction = HTML("<b>Визуализация</b>")

ui = VBox([instruction, eff_slider, fig, save_button, output_area])
display(ui)